## Overall Introduction of Innovative Methods

This part extends the official baseline (fixed train-test split) by testing four methods for imbalanced text classification:

1. **SMOTE + Logistic Regression** — data balancing  
2. **Weighted Ensemble(Random Forest + Gradient Boosting Tree + Logistic Regression)** — model combination  
3. **Threshold Moving** — decision threshold adjustment  
4. **Confidence-Based Pseudo-Labeling** — semi-supervised learning  

The purpose of these experiments is not only to improve overall classification performance, but also to investigate whether different strategies can better address data imbalance and enhance the recognition of minority-class samples.

In [1]:
!pip install dataproc-spark-connect==0.9.0 pyspark spark-nlp matplotlib graphframes

In [2]:
import math
import numpy as np
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, lit, when, lower, trim
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.linalg import Vectors, VectorUDT

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from imblearn.over_sampling import SMOTE

In [3]:
# =========================
# Create Spark Session
# =========================
spark = SparkSession.builder \
    .appName("CDS527_Innovation_Methods_Aligned_With_Baseline") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "8")

SEED = 42

In [4]:
# =========================
# Helper Functions
# =========================
def print_section_title(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

def extract_predictions_from_spark(pred_df, label_col="label", pred_col="prediction"):
    pdf = pred_df.select(label_col, pred_col).toPandas()
    y_true = pdf[label_col].astype(int).values
    y_pred = pdf[pred_col].astype(int).values
    return y_true, y_pred

def evaluate_and_print(method_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    print_section_title(method_name)
    print("Test Accuracy:", round(acc, 4))
    print("Test Precision:", round(precision, 4))
    print("Test Recall:", round(recall, 4))
    print("Test F1:", round(f1, 4))
    print("Confusion Matrix:")
    print(cm)

def get_metrics_dict(method_name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    recall = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "Method": method_name,
        "Accuracy": round(acc, 4),
        "Precision": round(precision, 4),
        "Recall": round(recall, 4),
        "F1": round(f1, 4),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1])
    }

def get_pos_prob(v):
    return float(v[1])

get_pos_prob_udf = udf(get_pos_prob, DoubleType())

def raw_to_prob(raw_vec):
    margin = float(raw_vec[1])
    return 1.0 / (1.0 + math.exp(-margin))

raw_to_prob_udf = udf(raw_to_prob, DoubleType())

prob_to_pred_udf = udf(lambda p: 1 if p >= 0.5 else 0, IntegerType())
array_to_vector_udf = udf(lambda arr: Vectors.dense(arr), VectorUDT())

# =========================
# 1. Load fixed split files
#    Expected columns: Review, Rating, Sentiment
# =========================
train_path = "train_fixed_split.csv"
test_path = "test_fixed_split.csv"

raw_train_df = spark.read.csv(train_path, header=True, inferSchema=True)
raw_test_df = spark.read.csv(test_path, header=True, inferSchema=True)

print_section_title("Loaded fixed split files")
print("Train size (raw):", raw_train_df.count())
print("Test size (raw):", raw_test_df.count())

print("\nTrain schema:")
raw_train_df.printSchema()

print("\nSample train rows:")
raw_train_df.show(5, truncate=False)

# =========================
# 2. Align columns with baseline format
#    Review -> text
#    Sentiment -> label
#    positive = 1, negative = 0
# =========================
train_df = raw_train_df.select(
    col("Review").cast("string").alias("text"),
    col("Rating"),
    when(lower(trim(col("Sentiment"))) == "positive", 1.0)
        .when(lower(trim(col("Sentiment"))) == "negative", 0.0)
        .otherwise(None)
        .alias("label")
)

test_df = raw_test_df.select(
    col("Review").cast("string").alias("text"),
    col("Rating"),
    when(lower(trim(col("Sentiment"))) == "positive", 1.0)
        .when(lower(trim(col("Sentiment"))) == "negative", 0.0)
        .otherwise(None)
        .alias("label")
)

train_df = train_df.dropna(subset=["text", "label"])
test_df = test_df.dropna(subset=["text", "label"])

print_section_title("After column alignment")
print("Train size (clean):", train_df.count())
print("Test size (clean):", test_df.count())

print("\nTrain label distribution:")
train_df.groupBy("label").count().show()

print("Test label distribution:")
test_df.groupBy("label").count().show()

# =========================
# 3. Build baseline-style TF-IDF features
# =========================
regex_tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="tokens",
    pattern="\\W+",
    toLowercase=True
)

stopword_remover = StopWordsRemover(
    inputCol="tokens",
    outputCol="filtered_tokens"
)

vectorizer = CountVectorizer(
    inputCol="filtered_tokens",
    outputCol="raw_features",
    vocabSize=10000,
    minDF=2.0
)

idf = IDF(
    inputCol="raw_features",
    outputCol="features"
)

feature_pipeline = Pipeline(stages=[
    regex_tokenizer,
    stopword_remover,
    vectorizer,
    idf
])

feature_model = feature_pipeline.fit(train_df)

train_feat = feature_model.transform(train_df).select("text", "label", "features")
test_feat = feature_model.transform(test_df).select("text", "label", "features")

print_section_title("Feature engineering completed")
print("Train feature count:", train_feat.count())
print("Test feature count:", test_feat.count())


Loaded fixed split files
Train size (raw): 1199
Test size (raw): 292

Train schema:
root
 |-- Review: string (nullable = true)
 |-- Rating: double (nullable = true)
 |-- Sentiment: string (nullable = true)


Sample train rows:
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
# =========================
# 4. Baseline placeholder
# =========================
results = []

official_baseline = {
    "Method": "Official_Baseline",
    "Accuracy": 0.8938,
    "Precision": 0.8800,
    "Recall": 0.8900,
    "F1": 0.8800,
    "TN": 4,
    "FP": 19,
    "FN": 11,
    "TP": 258
}

results.append(official_baseline)

print_section_title("Official Baseline (Placeholder)")
print("Test Accuracy:", official_baseline["Accuracy"])
print("Test Precision:", official_baseline["Precision"])
print("Test Recall:", official_baseline["Recall"])
print("Test F1:", official_baseline["F1"])

# Square confusion matrix
print("Confusion Matrix (square):")
print("          Predicted")
print("          positive  negative")
print(f"Actual positive     {official_baseline['TP']}       {official_baseline['FN']}")
print(f"      negative     {official_baseline['FP']}       {official_baseline['TN']}")

# A row of numerical form
print(f"Confusion Matrix (row): TP={official_baseline['TP']}, FN={official_baseline['FN']}, FP={official_baseline['FP']}, TN={official_baseline['TN']}")


Official Baseline (Placeholder)
Test Accuracy: 0.8938
Test Precision: 0.88
Test Recall: 0.89
Test F1: 0.88
Confusion Matrix (square):
          Predicted
          positive  negative
Actual positive     258       11
      negative     19       4
Confusion Matrix (row): TP=258, FN=11, FP=19, TN=4


In [6]:
# =========================
# Innovation 1: SMOTE + Logistic Regression
# =========================
print_section_title("Innovation 1 - SMOTE + Logistic Regression")

train_pd = train_feat.select("label", "features").toPandas()
X_train_np = np.array([x.toArray() for x in train_pd["features"]])
y_train_np = train_pd["label"].astype(int).values

print("Class distribution before SMOTE:")
unique, counts = np.unique(y_train_np, return_counts=True)
print(dict(zip(unique, counts)))

smote = SMOTE(random_state=SEED)
X_resampled, y_resampled = smote.fit_resample(X_train_np, y_train_np)

print("Class distribution after SMOTE:")
unique_res, counts_res = np.unique(y_resampled, return_counts=True)
print(dict(zip(unique_res, counts_res)))

smote_pd = pd.DataFrame({
    "label": y_resampled.tolist(),
    "features_array": [x.tolist() for x in X_resampled]
})

smote_spark = spark.createDataFrame(smote_pd)

smote_train_feat = smote_spark.withColumn(
    "features", array_to_vector_udf(col("features_array"))
).select(
    col("label").cast("double"),
    col("features")
)

lr_smote = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

smote_model = lr_smote.fit(smote_train_feat)
smote_pred = smote_model.transform(test_feat)

y_true_smote, y_pred_smote = extract_predictions_from_spark(smote_pred)
evaluate_and_print("Innovation 1 - SMOTE + Logistic Regression", y_true_smote, y_pred_smote)
results.append(get_metrics_dict("Innovation1_SMOTE_LR", y_true_smote, y_pred_smote))


Innovation 1 - SMOTE + Logistic Regression
Class distribution before SMOTE:
{0: 95, 1: 1104}
Class distribution after SMOTE:
{0: 1104, 1: 1104}

Innovation 1 - SMOTE + Logistic Regression
Test Accuracy: 0.8767
Test Precision: 0.8716
Test Recall: 0.8767
Test F1: 0.8741
Confusion Matrix:
[[  4  19]
 [ 17 252]]


In [7]:
# =========================
# Innovation 2: Weighted Ensemble
# =========================
print_section_title("Innovation 2 - Weighted Ensemble (RF + GBT + LR)")

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="rf_prediction",
    probabilityCol="rf_probability",
    numTrees=120,
    maxDepth=8,
    seed=SEED
)

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    predictionCol="gbt_prediction",
    maxIter=60,
    maxDepth=5,
    seed=SEED
)

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="lr_prediction",
    probabilityCol="lr_probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

rf_model = rf.fit(train_feat)
gbt_model = gbt.fit(train_feat)
lr_model = lr.fit(train_feat)

ensemble_df = test_feat

ensemble_df = rf_model.transform(ensemble_df)
ensemble_df = ensemble_df.withColumnRenamed("rawPrediction", "rf_rawPrediction")

ensemble_df = gbt_model.transform(ensemble_df)
ensemble_df = ensemble_df.withColumnRenamed("rawPrediction", "gbt_rawPrediction")

ensemble_df = lr_model.transform(ensemble_df)
ensemble_df = ensemble_df.withColumnRenamed("rawPrediction", "lr_rawPrediction")

ensemble_df = ensemble_df \
    .withColumn("rf_pos_prob", get_pos_prob_udf(col("rf_probability"))) \
    .withColumn("lr_pos_prob", get_pos_prob_udf(col("lr_probability"))) \
    .withColumn("gbt_pos_prob", raw_to_prob_udf(col("gbt_rawPrediction")))

w_rf = 0.35
w_gbt = 0.40
w_lr = 0.25

ensemble_df = ensemble_df.withColumn(
    "ensemble_prob",
    w_rf * col("rf_pos_prob") + w_gbt * col("gbt_pos_prob") + w_lr * col("lr_pos_prob")
)

ensemble_df = ensemble_df.withColumn(
    "prediction",
    prob_to_pred_udf(col("ensemble_prob"))
)

y_true_ens, y_pred_ens = extract_predictions_from_spark(ensemble_df)
evaluate_and_print("Innovation 2 - Weighted Ensemble (RF + GBT + LR)", y_true_ens, y_pred_ens)
results.append(get_metrics_dict("Innovation2_Weighted_Ensemble", y_true_ens, y_pred_ens))


Innovation 2 - Weighted Ensemble (RF + GBT + LR)

Innovation 2 - Weighted Ensemble (RF + GBT + LR)
Test Accuracy: 0.9007
Test Precision: 0.8597
Test Recall: 0.9007
Test F1: 0.878
Confusion Matrix:
[[  1  22]
 [  7 262]]


In [8]:
# =========================
# Innovation 3: Threshold Moving
# =========================
print_section_title("Innovation 3 - Threshold Moving")

train_sub, valid_sub = train_feat.randomSplit([0.8, 0.2], seed=SEED)

lr_threshold = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

threshold_model = lr_threshold.fit(train_sub)
valid_pred = threshold_model.transform(valid_sub)

valid_pdf = valid_pred.select("label", "probability").toPandas()
valid_pdf["pos_prob"] = valid_pdf["probability"].apply(lambda v: float(v[1]))

best_threshold = 0.50
best_f1 = -1

for threshold in np.arange(0.30, 0.71, 0.02):
    y_valid_true = valid_pdf["label"].astype(int).values
    y_valid_pred = (valid_pdf["pos_prob"] >= threshold).astype(int)
    current_f1 = f1_score(y_valid_true, y_valid_pred, average="weighted", zero_division=0)

    if current_f1 > best_f1:
        best_f1 = current_f1
        best_threshold = threshold

print(f"Best threshold found on validation set: {best_threshold:.2f}")
print(f"Best validation F1: {best_f1:.4f}")

final_threshold_model = lr_threshold.fit(train_feat)
test_threshold_pred = final_threshold_model.transform(test_feat)

test_threshold_pdf = test_threshold_pred.select("label", "probability").toPandas()
test_threshold_pdf["pos_prob"] = test_threshold_pdf["probability"].apply(lambda v: float(v[1]))
test_threshold_pdf["prediction"] = (test_threshold_pdf["pos_prob"] >= best_threshold).astype(int)

y_true_threshold = test_threshold_pdf["label"].astype(int).values
y_pred_threshold = test_threshold_pdf["prediction"].astype(int).values

evaluate_and_print(
    f"Innovation 3 - Threshold Moving (threshold={best_threshold:.2f})",
    y_true_threshold,
    y_pred_threshold
)

results.append(get_metrics_dict("Innovation3_Threshold_Moving", y_true_threshold, y_pred_threshold))


Innovation 3 - Threshold Moving
Best threshold found on validation set: 0.30
Best validation F1: 0.8015

Innovation 3 - Threshold Moving (threshold=0.30)
Test Accuracy: 0.726
Test Precision: 0.8895
Test Recall: 0.726
Test F1: 0.7864
Confusion Matrix:
[[ 13  10]
 [ 70 199]]


In [9]:
# =========================
# Innovation 4: Confidence-Based Pseudo-Labeling
# =========================
print_section_title("Innovation 4 - Confidence-Based Pseudo-Labeling")

labeled_df, pseudo_pool_df = train_df.randomSplit([0.8, 0.2], seed=SEED)

print("labeled_df size:", labeled_df.count())
print("pseudo_pool_df size:", pseudo_pool_df.count())
print("test_df size:", test_df.count())

pseudo_pipeline = Pipeline(stages=[
    RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+", toLowercase=True),
    StopWordsRemover(inputCol="tokens", outputCol="filtered_tokens"),
    CountVectorizer(inputCol="filtered_tokens", outputCol="raw_features", vocabSize=10000, minDF=2.0),
    IDF(inputCol="raw_features", outputCol="features")
])

pseudo_feature_model = pseudo_pipeline.fit(labeled_df)

labeled_feat = pseudo_feature_model.transform(labeled_df).select("text", "label", "features")
pseudo_pool_feat = pseudo_feature_model.transform(pseudo_pool_df).select("text", "features")
pseudo_test_feat = pseudo_feature_model.transform(test_df).select("label", "features")

pseudo_lr_init = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

initial_model = pseudo_lr_init.fit(labeled_feat)

pseudo_pred = initial_model.transform(pseudo_pool_feat)
pseudo_pred = pseudo_pred.withColumn("pos_prob", get_pos_prob_udf(col("probability")))

POS_THRESHOLD = 0.95
NEG_THRESHOLD = 0.05

high_conf_pos = pseudo_pred.filter(col("pos_prob") >= POS_THRESHOLD) \
    .select("text", "features") \
    .withColumn("final_label", lit(1.0))

high_conf_neg = pseudo_pred.filter(col("pos_prob") <= NEG_THRESHOLD) \
    .select("text", "features") \
    .withColumn("final_label", lit(0.0))

pseudo_selected = high_conf_pos.unionByName(high_conf_neg)

print("Selected pseudo-labeled samples:", pseudo_selected.count())
print("Pseudo positive count:", high_conf_pos.count())
print("Pseudo negative count:", high_conf_neg.count())

original_labeled_train = labeled_feat.select(
    "text",
    "features",
    col("label").alias("final_label")
)

augmented_train = original_labeled_train.unionByName(pseudo_selected)

print("Original labeled training size:", original_labeled_train.count())
print("Augmented training size:", augmented_train.count())

pseudo_lr = LogisticRegression(
    featuresCol="features",
    labelCol="final_label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxIter=100,
    regParam=0.0,
    elasticNetParam=0.0
)

pseudo_model = pseudo_lr.fit(augmented_train)

test_eval_df = pseudo_test_feat.select(
    "features",
    col("label").alias("final_label")
)

test_pred = pseudo_model.transform(test_eval_df).select("final_label", "prediction")

rows = test_pred.collect()
y_true_pseudo = [int(row["final_label"]) for row in rows]
y_pred_pseudo = [int(row["prediction"]) for row in rows]

evaluate_and_print(
    f"Innovation 4 - Confidence-Based Pseudo-Labeling ({POS_THRESHOLD:.2f}/{NEG_THRESHOLD:.2f})",
    y_true_pseudo,
    y_pred_pseudo
)

results.append(
    get_metrics_dict(
        "Innovation4_PseudoLabeling",
        y_true_pseudo,
        y_pred_pseudo
    )
)


Innovation 4 - Confidence-Based Pseudo-Labeling
labeled_df size: 991
pseudo_pool_df size: 208
test_df size: 292
Selected pseudo-labeled samples: 206
Pseudo positive count: 149
Pseudo negative count: 57
Original labeled training size: 991
Augmented training size: 1197

Innovation 4 - Confidence-Based Pseudo-Labeling (0.95/0.05)
Test Accuracy: 0.6986
Test Precision: 0.8773
Test Recall: 0.6986
Test F1: 0.766
Confusion Matrix:
[[ 11  12]
 [ 76 193]]


In [10]:
# =========================
# Final Comparison Table
# =========================
print_section_title("Final Comparison Table: Baseline vs Innovation Methods")

results_df = pd.DataFrame(results)
results_df = results_df.drop_duplicates(subset=["Method"]).reset_index(drop=True)
results_df = results_df.sort_values(by=["F1", "Accuracy"], ascending=False).reset_index(drop=True)

print(results_df)

results_df.to_csv("innovation_vs_baseline_results.csv", index=False, encoding="utf-8-sig")
print("\nSaved file: innovation_vs_baseline_results.csv")

try:
    from IPython.display import display
    display(results_df)
except:
    pass

# =========================
# Short Summary
# =========================
print_section_title("Short Summary")

best_method = results_df.iloc[0]["Method"]
best_f1 = results_df.iloc[0]["F1"]
best_acc = results_df.iloc[0]["Accuracy"]

print(f"The best-performing method in this comparison is: {best_method}")
print(f"Best F1: {best_f1}")
print(f"Best Accuracy: {best_acc}")


Final Comparison Table: Baseline vs Innovation Methods
                          Method  Accuracy  Precision  Recall      F1  TN  FP  \
0              Official_Baseline    0.8938     0.8800  0.8900  0.8800   4  19   
1  Innovation2_Weighted_Ensemble    0.9007     0.8597  0.9007  0.8780   1  22   
2           Innovation1_SMOTE_LR    0.8767     0.8716  0.8767  0.8741   4  19   
3   Innovation3_Threshold_Moving    0.7260     0.8895  0.7260  0.7864  13  10   
4     Innovation4_PseudoLabeling    0.6986     0.8773  0.6986  0.7660  11  12   

   FN   TP  
0  11  258  
1   7  262  
2  17  252  
3  70  199  
4  76  193  

Saved file: innovation_vs_baseline_results.csv


,Method,Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,Official_Baseline,0.8938,0.8800,0.8900,0.8800,4,19,11,258
1,Innovation2_Weighted_Ensemble,0.9007,0.8597,0.9007,0.8780,1,22,7,262
2,Innovation1_SMOTE_LR,0.8767,0.8716,0.8767,0.8741,4,19,17,252
3,Innovation3_Threshold_Moving,0.7260,0.8895,0.7260,0.7864,13,10,70,199
4,Innovation4_PseudoLabeling,0.6986,0.8773,0.6986,0.7660,11,12,76,193



Short Summary
The best-performing method in this comparison is: Official_Baseline
Best F1: 0.88
Best Accuracy: 0.8938


## Final Results Discussion

## Final Summary

Among all evaluated methods, the **Official_Baseline** demonstrated the strongest overall performance, achieving the best **Precision (0.88)** and **F1-score (0.88)**, while also maintaining highly competitive **Accuracy (0.8938)** and **Recall (0.89)**. These results suggest that the baseline provides the most balanced and reliable classification performance overall.

Although **Innovation2_Weighted_Ensemble** achieved slightly higher **Accuracy (0.9007)** and **Recall (0.9007)**, its weaker **Precision (0.8597)** and slightly lower **F1-score (0.878)** indicate that its overall performance is still slightly inferior to the **Official_Baseline**, especially in terms of balance and reliability across key evaluation metrics.

By comparison, **Innovation1_SMOTE_LR** showed moderate performance, while **Innovation3_Threshold_Moving** and **Innovation4_PseudoLabeling** performed worse overall due to lower recall and weaker balance across metrics.

Overall, the **Official_Baseline** remains the strongest method and the most suitable choice for final adoption.